# Bought or built?

Train the same LightGBM on different slices of the data to measure how much predictive power comes from the purchased external scores vs. engineered internal data. M0 naive form → M1 external-only → M2 engineered internal → M3 full.

In [ ]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from src.data import load_selected, RAW
from src.models import oof

INTERIM = Path("../data/interim")
X, y = load_selected()
bp = json.loads((INTERIM / "best_params.json").read_text()) if (INTERIM / "best_params.json").exists() else {}
P = {"n_estimators": 800, "learning_rate": 0.03, "num_leaves": 31, "min_child_samples": 50,
     "subsample": 0.8, "colsample_bytree": 0.7, "reg_lambda": 2.0, **bp.get("lgb", {})}

def gbm():
    return lgb.LGBMClassifier(**P, subsample_freq=1, n_jobs=-1, verbose=-1)

ext = [c for c in X.columns if c.lower().startswith("ext_") or "ext_source" in c.lower()]
internal = [c for c in X.columns if c not in ext]
ext

## The ladder

In [ ]:
app = pd.read_csv(RAW / "application_train.csv").set_index("SK_ID_CURR")
naive_cols = [c for c in app.select_dtypes("number").columns if c != "TARGET" and "EXT_SOURCE" not in c]
models = {
    "M0 naive form":          app.loc[X.index, naive_cols],
    "M1 external only":       X[ext],
    "M2 engineered internal": X[internal],
    "M3 full":                X,
}
oofs = {}
for name, Xi in models.items():
    oofs[name] = oof(gbm(), Xi, y)
    print(f"{name:24s} {Xi.shape[1]:4d} feats   AUC {roc_auc_score(y, oofs[name]):.4f}")

## Gaps

In [ ]:
a = {k: roc_auc_score(y, v) for k, v in oofs.items()}
print(f"buy the scores      M1-M0  {a['M1 external only']-a['M0 naive form']:+.4f}")
print(f"build internal      M2-M0  {a['M2 engineered internal']-a['M0 naive form']:+.4f}")
print(f"external marginal   M3-M2  {a['M3 full']-a['M2 engineered internal']:+.4f}")
print(f"built vs bought     M2-M1  {a['M2 engineered internal']-a['M1 external only']:+.4f}")

## Thin vs thick files

In [ ]:
full, internal_p = oofs["M3 full"], oofs["M2 engineered internal"]
thickness = X[internal].notna().sum(axis=1)
thin = (thickness < thickness.median()).to_numpy()
yv = y.to_numpy()
for label, mask in [("thin", thin), ("thick", ~thin)]:
    af, ai = roc_auc_score(yv[mask], full[mask]), roc_auc_score(yv[mask], internal_p[mask])
    print(f"{label:5s} n={int(mask.sum()):6d}   full {af:.4f}   internal {ai:.4f}   external lift {af-ai:+.4f}")